# exp_08b Proximity Ablation


In [ ]:
# Install package from GitHub
!pip install git+https://github.com/SilasPignotti/urban-tree-transfer.git -q
# Optional dependencies
!pip install optuna pytorch-tabnet -q

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive

drive.mount('/content/drive')

BASE_DIR = Path('/content/drive/MyDrive/dev/urban-tree-transfer')
DATA_DIR = BASE_DIR / 'data'
OUTPUT_DIR = BASE_DIR / 'outputs/phase_3'

(OUTPUT_DIR / 'metadata').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'figures').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'logs').mkdir(parents=True, exist_ok=True)

from urban_tree_transfer.config import load_experiment_config
from urban_tree_transfer.experiments import (
    ablation,
    data_loading,
    preprocessing,
    training,
    evaluation,
    visualization,
    models,
    transfer,
    hp_tuning,
)
from urban_tree_transfer.utils import (
    validate_setup_decisions,
    validate_algorithm_comparison,
    validate_hp_tuning_result,
    validate_evaluation_metrics,
    validate_finetuning_curve,
)


In [ ]:
config = load_experiment_config()
variants = config['setup_ablation']['proximity']['variants']

results = []
for variant in variants:
    train_df, val_df, _ = data_loading.load_berlin_splits(
        DATA_DIR / 'phase_2_splits', variant=variant['name']
    )
    feature_cols = data_loading.get_feature_columns(train_df)
    x_train_scaled, x_val_scaled, _, _ = preprocessing.scale_features(
        train_df[feature_cols], x_val=val_df[feature_cols]
    )
    y_train, label_to_idx, _ = preprocessing.encode_genus_labels(train_df['genus_latin'])
    y_val = val_df['genus_latin'].map(label_to_idx).to_numpy()
    cv = training.create_spatial_block_cv(train_df, n_splits=config['global']['cv_folds'])

    rf = models.create_model('random_forest')
    metrics = training.train_with_cv(
        rf, x_train_scaled, y_train, train_df['block_id'].values, cv
    )
    results.append({
        'variant': variant['name'],
        'val_f1_mean': metrics['val_f1_mean'],
        'val_f1_std': metrics['val_f1_std'],
        'train_val_gap': metrics['train_val_gap'],
        'samples_retained': len(train_df),
    })

results_df = pd.DataFrame(results)
fig_dir = OUTPUT_DIR / 'figures/exp_08b_proximity'
fig_dir.mkdir(parents=True, exist_ok=True)
visualization.plot_ablation_comparison(
    results_df,
    title='Proximity Filter Comparison',
    output_path=fig_dir / 'proximity_ablation.png',
)
